# Target Trial Emulation (TTE)

## What is TTE?

**Target Trial Emulation (TTE)** is a framework for **designing and conducting statistical analyses** to assess the effectiveness of medical treatments using **observational data**.

Observational data refers to **real-world data** collected from sources such as:
- **Electronic Health Records (EHRs)**
- **Patient Registries**
- **Administrative Claims Databases**
- **Cohort Studies and Surveys**

These data sources are used because:

- They **span long time periods**, allowing the study of long-term effects.
- They **enable studies that would be unethical as RCTs** (e.g., smoking, pollution exposure).
- They **capture a broader population**, unlike RCTs that may have strict inclusion criteria.

When **properly implemented on high-quality observational data**, TTE provides a **credible method** to determine treatment effectiveness while mitigating biases.

## Purpose of TTE

The purpose of **Target Trial Emulation (TTE)** is to **reduce biases** commonly found in **observational studies of medical treatments**. Observational studies often suffer from **flawed study designs**, where treatment assignment and follow-up are **not properly aligned at time zero**, failing to mimic the design of a **randomized controlled trial (RCT)**. These misalignments introduce biases such as **immortal time bias** and **confounding**, which can lead to incorrect conclusions about a treatments effectiveness.

By **structuring observational analyses as if they were an RCT**, TTE provides a framework to **estimate causal effects** more reliably. This approach ensures that findings are **more directly useful for clinical decision-making and policy development**.

# Libraries/modules used

The module `TrialEmulation_python` is the Python code we converted from the TrialEmulation package of R.

In [3]:
import TrialEmulation_python as ttepy
import statsmodels.api as sm
import os
import tempfile
import pandas as pd

# Code 

The TrialEmulation package of R was used for converting relevant R code for the classes, functions, and methods used in TTE to Python.

## 1. Setup

A sequence of target trials analysis starts by specifying which estimand will be used:

In [4]:
# Initialize trial sequences for different estimands
trial_pp = ttepy.trial_sequence(estimand="PP")  # Per-protocol
trial_itt = ttepy.trial_sequence(estimand="ITT")  # Intention-to-treat



Additionally it is useful to create a directory to save files for later inspection.

In [5]:
trial_pp_dir = os.path.join(os.getcwd(), "trial_pp")
os.makedirs(trial_pp_dir, exist_ok=True)

trial_itt_dir = os.path.join(os.getcwd(), "trial_itt")
os.makedirs(trial_itt_dir, exist_ok=True)

## 2. Data preparation

In [6]:
data = pd.read_csv("data_censored.csv")
#display(data)
trial_pp.set_data(
    data=data,
    ID="id", 
    period="period", 
    treatment="treatment", 
    outcome="outcome", 
    eligible="eligible"
)

trial_itt.set_data(
    data=data,
    ID="id", 
    period="period", 
    treatment="treatment", 
    outcome="outcome", 
    eligible="eligible"
)
trial_itt.show()


Trial Sequence Object
Estimand: ITT

Data:
 - N: 725 observations from 89 patients


/home/ian/school/data_analytics/Assignments/clustering_assignment_1/TrialEmulation_python.py:195: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '1' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.
  sw_data.loc[switch_mask & (treatment == 1), ["started1", "eligible1_sw"]] = 1
/home/ian/school/data_analytics/Assignments/clustering_assignment_1/TrialEmulation_python.py:197: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '1' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.
  sw_data.loc[switch_mask & (treatment == 0), ["started0", "eligible0_sw"]] = 1


,id,period,treatment,x1,x2,x3,x4,age,age_s,outcome,censored,eligible,time_on_regime
0,1,0,1,1,1.146148,0,0.734203,36,0.083333,0,0,1,0.0
1,1,1,1,1,0.002200,0,0.734203,37,0.166667,0,0,0,1.0
2,1,2,1,0,-0.481762,0,0.734203,38,0.250000,0,0,0,2.0
3,1,3,1,0,0.007872,0,0.734203,39,0.333333,0,0,0,3.0
4,1,4,1,1,0.216054,0,0.734203,40,0.416667,0,0,0,4.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
719,99,2,0,0,1.650478,1,0.575268,67,2.666667,0,0,0,2.0
720,99,3,0,0,-0.747906,1,0.575268,68,2.750000,0,0,0,1.0
721,99,4,0,0,-0.790056,1,0.575268,69,2.833333,0,0,0,2.0
722,99,5,1,1,0.387429,1,0.575268,70,2.916667,0,0,0,3.0



IPW for informative censoring:
 - No weight model specified

Sequence of Trials Data:
- Use set_expansion_options() and expand_trials() to construct the sequence of trials dataset.

Outcome model:
 - Outcome model not specified. Use set_outcome_model()



## 3. Weight Models and Censoring

### 3.1 Censoring due to treatment switching

We specify model formulas to be used for calculating the probability of receiving treatment in the current period. Separate models are fitted for patients who had treatment = 1 and those who had treatment = 0 in the previous period. Stabilized weights are used by fitting numerator and denominator models.

There are optional arguments to specify columns which can include/exclude observations from the treatment models. These are used in case it is not possible for a patient to deviate from a certain treatment assignment in that period.

In [7]:
# Call set_switch_weight_model
trial_pp.set_switch_weight_model(
    numerator="~age",
    denominator="~age + x1 + x3",
    model_fitter=ttepy.stats_glm_logit(save_path=os.path.join(trial_pp_dir, "switch_models"))
)

# Access and inspect switch_weights (equivalent to trial_pp@switch_weights)
trial_pp.switch_weights.show()

 - Numerator formula: treatment ~ age
 - Denominator formula: treatment ~ age + x1 + x3
 - Model fitter type: te_stats_glm_logit
 - Weight models not fitted. Use calculate_weights()


### 3.2 Other informative censoring

In case there is other informative censoring occurring in the data, we can create similar models to estimate the IPCW. These can be used with all types of estimand. We need to specifycensor_event which is the column containing the censoring indicator.


In [8]:
trial_pp.set_censor_weight_model(
    censor_event="censored",
    numerator="1 - censored ~ x2",
    denominator="1 - censored ~ x2 + x1",
    model_fitter=ttepy.stats_glm_logit(save_path=os.path.join(trial_pp_dir, "switch_models"))
)

trial_pp.censor_weights.show()

 - Numerator formula: treatment ~ 1 - censored ~ x2
 - Denominator formula: treatment ~ 1 - censored ~ x2 + x1
 - Model fitter type: te_stats_glm_logit
 - Weight models not fitted. Use calculate_weights()


In [9]:
trial_itt.set_censor_weight_model(
    censor_event="censored",
    numerator="1 - censored ~ x2",
    denominator="1 - censored ~ x2 + x1",
    pool_models="numerator",  # ITT uses pooling
    model_fitter=ttepy.stats_glm_logit(save_path=os.path.join(trial_itt_dir, "switch_models"))
)
trial_itt.censor_weights.show()

 - Numerator formula: treatment ~ 1 - censored ~ x2
 - Denominator formula: treatment ~ 1 - censored ~ x2 + x1
 - Numerator model is pooled across treatment arms. Denominator model is not pooled.
 - Model fitter type: te_stats_glm_logit
 - Weight models not fitted. Use calculate_weights()


## 4. Calculate weights

Next we need to fit the individual models and combine them into weights. This is done with calculate_weights().

In [10]:
import pickle  
# Calculate Weights
trial_pp.calculate_weights(save_path=trial_pp_dir)
trial_itt.calculate_weights(save_path=trial_itt_dir)

# Show weight models (equivalent to show_weight_models(trial_itt) in R)
trial_itt.show_weight_models()

⚠ Warning: No data for previous_treatment = 1. Skipping model fit.
Fitting treatment switching models...
## Weight Models for Informative Censoring
## ---------------------------------------

## [[numerator]]
Model: P(censor_event = 0 | X)

     term     Coef.  Std.Err.          z        P>|z|    [0.025    0.975]
Intercept -2.448091  0.140575 -17.414876 6.362614e-68 -2.723612 -2.172569
       x2  0.448648  0.136878   3.277724 1.046476e-03  0.180372  0.716924


 null.deviance df.null logLik    AIC      BIC      deviance df.residual nobs
 404.2156      724     -196.7002 397.4004 406.5727 393.4004 723.0         725 

 path
 /home/ian/school/data_analytics/Assignments/clustering_assignment_1/itt_models/model_numerator.pkl

## [[denominator_0]]
Model: P(censor_event = 0 | X, previous treatment = 0)

     term     Coef.  Std.Err.         z        P>|z|    [0.025    0.975]
Intercept -2.000404  0.204162 -9.798137 1.146813e-22 -2.400554 -1.600255
       x2  0.599006  0.175156  3.419850 6.265558

In [11]:
trial_pp.show_weight_models()

## Weight Models for Informative Censoring
## ---------------------------------------

## [[numerator]]
Model: P(censor_event = 0 | X)

     term     Coef.  Std.Err.         z        P>|z|    [0.025    0.975]
Intercept -1.402654  0.199368 -7.035519 1.985211e-12 -1.793407 -1.011901
       x2  0.543659  0.207566  2.619218 8.813164e-03  0.136838  0.950480


 null.deviance df.null logLik    AIC      BIC      deviance df.residual nobs
 172.8729      169     -82.8135 169.6270 175.8986 165.6270 168.0         170 

 path
 /home/ian/school/data_analytics/Assignments/clustering_assignment_1/pp_models/model_numerator.pkl

## [[denominator_0]]
Model: P(censor_event = 0 | X, previous treatment = 0)

     term     Coef.  Std.Err.         z    P>|z|    [0.025    0.975]
Intercept -1.033790  0.244917 -4.220979 0.000024 -1.513819 -0.553761
       x2  0.618956  0.215314  2.874659 0.004045  0.196947  1.040964
       x1 -0.945399  0.422381 -2.238261 0.025204 -1.773250 -0.117547


 null.deviance df.null log

## 5. Specify Outcome Model

Now we can specify the outcome model. Here we can include adjustment terms for any variables in the dataset. The numerator terms from the stabilised weight models are automatically included in the outcome model formula.

In [12]:
# Set outcome model for Per-Protocol analysis (PP)
trial_pp.set_outcome_model()

# Set outcome model for Intention-To-Treat analysis (ITT) with adjustment terms
trial_itt.set_outcome_model(adjustment_terms="~x2")

## 6. Expand Trials

Now we are ready to create the data set with all of the sequence of target trials.

In [13]:
# Set expansion options for Per-Protocol analysis (PP)
trial_pp.set_expansion_options(
    output="save_to_datatable",  # Placeholder for equivalent functionality
    chunk_size=500  # The number of patients per expansion iteration
)

# Set expansion options for Intention-To-Treat analysis (ITT)
trial_itt.set_expansion_options(
    output="save_to_datatable",
    chunk_size=500
)

trial_pp = trial_pp.set_expansion_options(output="save_to_datatable", chunk_size=500)
trial_itt = trial_itt.set_expansion_options(output="save_to_datatable", chunk_size=500)


## 6.1 Create Sequence of Trials Data


In [14]:
# Set expansion options before expanding trials
trial_pp.set_expansion_options(output="datatable", chunk_size=500)
trial_itt.set_expansion_options(output="datatable", chunk_size=500)

# Expand trials (Stores Data)
trial_pp.expand_trials()
trial_itt.expand_trials()

# Print the expansion data
trial_pp.show_expansion()

## Sequence of Trials Data:
## - Chunk size: 500
## - Censor at switch: TRUE
## - First period: 0 | Last period: Inf

## A TE Datastore Datatable object
## N: 340 observations

## id <int>  trial_period <int>  followup_time <int>  outcome <num>  weight <num>  treatment <num>  x2 <num>  age <num>  assigned_treatment <num>

 id  trial_period  followup_time  outcome  weight  treatment        x2  age  assigned_treatment
  1             0              0      0.0     1.0          1  1.146148   36                   1
  1             0              1      0.0     0.9          1  1.146148   36                   1
  2             0              0      0.0     1.0          0 -0.802142   26                   0
  2             0              0      0.0     1.0          0 -0.802142   26                   0
  2             0              1      0.0     1.1          0 -0.802142   26                   0
   ---
 id  trial_period  followup_time  outcome  weight  treatment        x2  age  assigned_treatme

## 7. Load or Sample from Expanded Data

Now that the expanded data has been created, we can prepare the data to fit the outcome model. For data that can fit comfortably in memory, this is a trivial step using load_expanded_data.

For large datasets, it may be necessary to sample from the expanded by setting the p_control argument. This sets the probability that an observation with outcome == 0 will be included in the loaded data. A seed can be set for reproducibility. Additionally, a vector of periods to include can be specified, e.g., period = 1:60, and/or a subsetting condition, subset_condition = "age > 65".

In [15]:
# Ensure expansion options are set
trial_itt.set_expansion_options(output="datatable", chunk_size=500)

# Now expand trials
trial_itt.expand_trials()

# Then load expanded data
trial_itt.load_expanded_data(seed=1234, p_control=0.5)

## 8. Fit Marginal Structural Model
To fit the outcome model we use fit_msm()

In [16]:
trial_itt.fit_msm() 


# ** Print the Outcome Model (Equivalent to `trial_itt@outcome_model` in R) **
print(f"## - Formula: {trial_itt.outcome_model['formula']}")
print(f"## - Treatment variable: {trial_itt.outcome_model['treatment_variable']}")
print(f"## - Adjustment variables: {', '.join(trial_itt.outcome_model['adjustment_variables'])}")
print(f"## - Model fitter type: {trial_itt.outcome_model['model_fitter']}")

# ** Display Model Summary (Equivalent to R Output) **
print("\n## Model Summary:\n")
print(trial_itt.outcome_model["fitted_model"].summary())  

## - Formula: outcome ~ assigned_treatment + x2 + followup_time + I(followup_time^2) + trial_period + I(trial_period^2)
## - Treatment variable: assigned_treatment
## - Adjustment variables: x2
## - Model fitter type: te_stats_glm_logit

## Model Summary:

 estimate  std.error  statistic  p.value  conf.low  conf.high
    -0.71       0.23      -3.07 2.15e-03     -1.16      -0.25
    -0.39       0.98      -0.40     0.69     -2.31       1.53
     1.44       0.78       1.85 6.47e-02 -8.76e-02       2.97
     0.66       0.70       0.94     0.35     -0.71       2.02
    -0.76       0.31      -2.43 1.49e-02     -1.36      -0.15
 0.00e+00   0.00e+00        NaN      NaN  0.00e+00   0.00e+00
    -1.41       0.46      -3.07 2.15e-03     -2.31      -0.51

##
##  logLik     AIC  BIC  deviance  df_residual  nobs
##  -20.5  49  -3053  41.0  498  500
## - Formula: outcome ~ assigned_treatment + x2 + followup_time + I(followup_time^2) + trial_period + I(trial_period^2)
## - Treatment variable: assigned

/home/ian/.local/lib/python3.10/site-packages/statsmodels/genmod/generalized_linear_model.py:1923: FutureWarning: The bic value is computed using the deviance formula. After 0.13 this will change to the log-likelihood based formula. This change has no impact on the relative rank of models compared using BIC. You can directly access the log-likelihood version using the `bic_llf` attribute. You can suppress this message by calling statsmodels.genmod.generalized_linear_model.SET_USE_BIC_LLF with True to get the LLF-based version now or False to retainthe deviance version.
  warnings.warn(


Depending on the model fitter used, we can also access the model object. For the default stats::glm logistic model, we have the glm object as well as the sandwich variance-covariance matrix.

In [17]:
# Extract the fitted MSM model from outcome_model
msm_model = trial_itt.outcome_model["fitted_model"]

# Print the equivalent R-style model summary
print("##")
print("## Call:  glm(formula = formula, family = binomial('logit'), data = data, ")
print("##     weights = weights, x = FALSE, y = FALSE)")
print("##")

# Extract Coefficients
coef_summary = msm_model.summary2().tables[1]
coef_summary = coef_summary.rename(columns={
    "Coef.": "estimate",
    "Std.Err.": "std.error",
    "z": "statistic",
    "P>|z|": "p.value",
    "[0.025": "conf.low",
    "0.975]": "conf.high"
})

# Print Coefficients (Formatted to Match R Output)
print("## Coefficients:")
for index, row in coef_summary.iterrows():
    print(f"## {index:>20} {row['estimate']:>10.5f}")

print("##")
print(f"## Degrees of Freedom: {int(msm_model.nobs)} Total (i.e. Null);  {int(msm_model.df_resid)} Residual")
print(f"## Null Deviance:       {msm_model.null_deviance:.1f}")
print(f"## Residual Deviance:   {msm_model.deviance:.1f}     AIC: {msm_model.aic:.1f}")


##
## Call:  glm(formula = formula, family = binomial('logit'), data = data, 
##     weights = weights, x = FALSE, y = FALSE)
##
## Coefficients:
##            Intercept   -0.70550
##   assigned_treatment   -0.38890
##                   x2    1.44030
##        followup_time    0.65575
## I(followup_time ^ 2)   -0.75525
##         trial_period    0.00000
##  I(trial_period ^ 2)   -1.41100
##
## Degrees of Freedom: 500 Total (i.e. Null);  497 Residual
## Null Deviance:       44.7
## Residual Deviance:   41.0     AIC: 49.0


trial_itt@outcome_model@fitted@model$vcov

In [18]:
# Extract the fitted MSM model
msm_model = trial_itt.outcome_model["fitted_model"]

# Extract the variance-covariance matrix
vcov_matrix = msm_model.cov_params()

# Print the formatted variance-covariance matrix to match R output
print("##")
print(vcov_matrix.to_string(float_format=lambda x: f"{x:.9f}"))


##
                        Intercept  assigned_treatment           x2  followup_time  I(followup_time ^ 2)  trial_period  I(trial_period ^ 2)
Intercept             0.052839721        -0.046867056 -0.044826561   -0.150063693          -0.044384250   0.000000000          0.105679443
assigned_treatment   -0.046867056         0.960774361 -0.089125261    0.015861077          -0.077873035   0.000000000         -0.093734112
x2                   -0.044826561        -0.089125261  0.607710853    0.040053449          -0.049599672   0.000000000         -0.089653121
followup_time        -0.150063693         0.015861077  0.040053449    0.485114000           0.184986614   0.000000000         -0.300127386
I(followup_time ^ 2) -0.044384250        -0.077873035 -0.049599672    0.184986614           0.096218115   0.000000000         -0.088768500
trial_period          0.000000000         0.000000000  0.000000000    0.000000000           0.000000000   0.000000000          0.000000000
I(trial_period ^ 2)   0.

# 9. Inference

We use the predict() method to estimate survival probabilities or cumulative incidences for different values of assigned_treatment.

In [19]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d

def predict(self, newdata=None, predict_times=None, type="survival"):
    if newdata is None:
        if self.expansion is None:
            raise ValueError("⚠ Expansion data not available. Run load_expanded_data() first.")
        data = self.expansion.copy()
    else:
        data = newdata.copy()
    
    if predict_times is None:
        predict_times = np.arange(0, 21)  # Default follow-up times from 0 to 20
    
    predict_times = np.array(predict_times, dtype=int)
    
    # Ensure model is fitted
    if "fitted_model" not in self.outcome_model or self.outcome_model["fitted_model"] is None:
        raise ValueError("⚠ Outcome model not fitted. Run fit_msm() first.")

    model = self.outcome_model["fitted_model"]  # Extract fitted model
    
    # Prepare control and treated groups
    data_control = data.copy()
    data_control["assigned_treatment"] = 0  # Control group
    
    data_treated = data.copy()
    data_treated["assigned_treatment"] = 1  # Treated group
    
    # Predict survival probabilities
    preds_control = model.predict(data_control)
    preds_treated = model.predict(data_treated)
    
    # Compute survival estimates (1 - probability of event)
    survival_control = 1 - pd.Series(preds_control).groupby(data_control["followup_time"]).mean()
    survival_treated = 1 - pd.Series(preds_treated).groupby(data_treated["followup_time"]).mean()
    
    # Compute survival differences
    survival_diff = (survival_treated.reindex(predict_times).ffill().fillna(0) - 
                     survival_control.reindex(predict_times).ffill().fillna(0))
    
    # Compute 95% Confidence Intervals (assuming normal approximation)
    ci_lower = survival_diff - 1.96 * np.std(survival_diff)
    ci_upper = survival_diff + 1.96 * np.std(survival_diff)
    
    # Generate a smoother curve using cubic interpolation
    fine_times = np.linspace(predict_times.min(), predict_times.max(), 100)
    
    interp_diff = interp1d(predict_times, survival_diff, kind='cubic', fill_value="extrapolate")
    interp_lower = interp1d(predict_times, ci_lower, kind='cubic', fill_value="extrapolate")
    interp_upper = interp1d(predict_times, ci_upper, kind='cubic', fill_value="extrapolate")
    
    smooth_diff = interp_diff(fine_times)
    smooth_lower = interp_lower(fine_times)
    smooth_upper = interp_upper(fine_times)
    
    # Plot survival difference over time
    plt.figure(figsize=(8, 5))
    plt.plot(fine_times, smooth_diff, label="Survival Difference", color="blue")
    plt.plot(fine_times, smooth_lower, linestyle='--', color="red", label="95% CI Lower Bound")
    plt.plot(fine_times, smooth_upper, linestyle='--', color="red", label="95% CI Upper Bound")
    
    plt.xlabel("Follow-up Time")
    plt.ylabel("Survival Difference")
    plt.title("Predicted Survival Difference Over Time")
    plt.legend()
    plt.grid()
    plt.show()

# Attach function to `TrialSequence`
TrialSequence.predict = predict


NameError: name 'TrialSequence' is not defined